# Why Autograd?

In deep learning, we need gradients (derivatives) to update weights during training.

For small equations, derivatives are easy.

Example:

y = x²

Derivative:

dy/dx = 2x

If x = 3:

dy/dx = 6

But neural networks contain many nested functions, so manual differentiation becomes difficult and error-prone.

Example:

y = x²  
z = sin(y)

Now:

dz/dx = dz/dy × dy/dx

This uses the chain rule.

Imagine doing this manually for millions of parameters in a neural network 😅

PyTorch Autograd automatically calculates these gradients for us.

## Example

```python
import torch

x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

y.backward()

print(x.grad)   # 6

# What is Autograd?

Autograd is a PyTorch feature that automatically calculates gradients (derivatives).

It tracks all operations performed on tensors and builds a **Computation Graph**.

Then during backpropagation, it uses the **chain rule** to compute gradients automatically.


# How Autograd Works

## Step 1: Create Tensor

```python
x = torch.tensor(2.0, requires_grad=True)
```

- `requires_grad=True`
→ tells PyTorch to track operations on `x`



## Step 2: Perform Operations

```python
y = x ** 2
z = y + 3
```

PyTorch creates a computation graph internally.

## Computation Graph

```text
x ───► y = x² ───► z = y + 3
```

Each node stores:
- operation performed
- relation between tensors


## Step 3: Backward Pass

```python
z.backward()
```

Autograd now moves backward through the graph.

```text
z ───► y ───► x
```

Using chain rule:

dz/dx = dz/dy × dy/dx


# Complete Example

```python
import torch

# Step 1
x = torch.tensor(2.0, requires_grad=True)

# Step 2
y = x ** 2
z = y + 3

# Step 3
z.backward()

print(x.grad)
```

## Output

```python
tensor(4.)
```

Because:

```text
z = x² + 3

dz/dx = 2x

x = 2

dz/dx = 4
```


# Simple Flow Diagram

```text
Forward Pass:
x → y=x² → z=y+3

Backward Pass:
z → y → x

Gradient Flow:
dz/dy × dy/dx
```


# Key Point

Autograd:
- tracks operations
- builds computation graph
- applies chain rule automatically
- computes gradients during `backward()`

This is the core of neural network training in PyTorch.

# Gradient Accumulation Problem in PyTorch

In PyTorch, gradients are **stored (accumulated)** by default.

This means every time we call:

```python
backward()
```

new gradients are **added** to old gradients instead of replacing them.


# Problem Example

```python
import torch

x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()

print(x.grad)   # tensor(4.)

# backward again
y = x ** 2
y.backward()

print(x.grad)   # tensor(8.)
```


# Why Did It Become 8?

First backward:

```text
dy/dx = 2x = 4
```

Gradient stored:

```text
x.grad = 4
```

Second backward:

New gradient:

```text
4
```

PyTorch adds it:

```text
4 + 4 = 8
```


# Visualization

```text
1st backward:
x.grad = 4

2nd backward:
x.grad = 4 + 4

Final:
x.grad = 8
```


# Why Does PyTorch Do This?

Gradient accumulation is useful in:
- mini-batch training
- large models
- gradient accumulation techniques

But during normal training, accumulated gradients can cause wrong updates.


# Solution → Clear Gradients

Before every new backward pass:

```python
x.grad.zero_()
```

or

```python
optimizer.zero_grad()
```


# Correct Way

```python
import torch

x = torch.tensor(2.0, requires_grad=True)

# First pass
y = x ** 2
y.backward()

print(x.grad)   # 4

# Clear old gradients
x.grad.zero_()

# Second pass
y = x ** 2
y.backward()

print(x.grad)   # 4
```


# Training Loop Flow

```text
Forward Pass
      ↓
Calculate Loss
      ↓
Backward Pass
      ↓
Update Weights
      ↓
Clear Gradients
      ↓
Next Iteration
```

# Key Point

PyTorch stores gradients by default:

```text
new_grad = old_grad + current_grad
```

So we must clear gradients before the next training step using:

```python
optimizer.zero_grad()
```

# Problem During Prediction (Inference)

During prediction/testing, we do NOT need gradients because:
- no training happens
- weights are not updated
- backpropagation is unnecessary

But by default, Autograd still tracks operations.

This causes:
- extra memory usage
- slower execution
- unnecessary computation graph creation

So during prediction, we disable Autograd.


# Example Problem

```python
import torch

x = torch.tensor(5.0, requires_grad=True)

y = x * 2

print(y)
```

Even during simple prediction, PyTorch creates a computation graph.

```text
x → y
```

This is unnecessary for inference.


# Solution → Disable Gradient Tracking

There are 3 common methods.


# 1. Using `torch.no_grad()`  (Most Common)

Temporarily disables Autograd.

```python
import torch

x = torch.tensor(5.0, requires_grad=True)

with torch.no_grad():
    y = x * 2

print(y)
```

## What Happens?

```text
No computation graph created
No gradient tracking
Less memory used
Faster prediction
```


# Diagram

```text
Training:
x → operation → graph created

Prediction using no_grad:
x → operation
(no graph)
```


# 2. Using `.detach()`

Detach removes a tensor from the computation graph.

```python
import torch

x = torch.tensor(5.0, requires_grad=True)

y = x * 2

z = y.detach()

print(z)
```

Now:

```text
z is disconnected from graph
```


# Diagram

```text
Before detach:
x → y

After detach:
x → y    z(detached)
```

`z` will not track gradients.


# Why Use Detach?

Useful when:
- converting tensor to NumPy
- inference
- stopping gradient flow in some layers

Example:

```python
arr = z.numpy()
```

Without detach, PyTorch may give error because NumPy does not support gradient tracking.


# 3. Using `requires_grad=False`

Disable gradient tracking permanently for a tensor.

```python
x = torch.tensor(5.0, requires_grad=False)
```

or

```python
x.requires_grad_(False)
```

Now Autograd ignores this tensor completely.


# When to Use Which?

| Method | Use Case |
|---|---|
| `torch.no_grad()` | Prediction / inference |
| `.detach()` | Remove tensor from graph |
| `requires_grad=False` | Tensor never needs gradients |


# Most Common in Real Models

```python
model.eval()

with torch.no_grad():
    prediction = model(x)
```

This is the standard prediction workflow in PyTorch.


# Key Idea

During training:
- gradients needed
- graph needed

During prediction:
- gradients NOT needed
- disable Autograd for better performance